# 00.4 — Matmul as composed dot products, and shape algebra

**Question:** given `(a, b) @ (c, d)`, can I say in **one second** whether it is legal and what shape comes out — without running it?

**Prereq:** 00.3 (dot product).

**Interview one-liners**
- A matmul is nothing but a **grid of dot products**. `C[i, j] = row i of A · column j of B`.
- The rule: **inner dims must match, inner dims vanish, outer dims survive.** `(m, k) @ (k, n) -> (m, n)`.
- Exact cost: `m · n · (2k − 1)` — each output cell is `k` multiplies + `k − 1` adds. Rounded, everyone quotes `2mkn`.
- Leading dimensions are **batches** — a free for-loop. Same weights, different data.
- The output's last dim comes from the **weight**, not from your data. `x @ W` with `W` of shape `(d_in, d_out)` maps features `d_in -> d_out` and leaves batch/time dims untouched.

In [1]:
import sys
sys.path.insert(0, '..')

import torch
from common import show, shape_table

torch.manual_seed(0)
print('torch', torch.__version__)

torch 2.10.0


## Experiment 1 — a matmul really is just dot products

Square matrices hide the rule, because every dimension is the same number and you
cannot tell which one came from where. So use `A` of shape `(2, 3)` and `B` of shape `(3, 4)`.

We compute `A @ B` **two ways** and check they agree:
1. `torch.matmul` — what PyTorch does.
2. A double Python loop filling `C[i, j] = A[i] · B[:, j]` — one dot product per cell.

If they match, then a matmul contains no magic beyond 00.3.

In [2]:
# Known integer entries so you can check any cell by hand.
A = torch.arange(1., 7.).reshape(2, 3)     # [[1,2,3],
                                           #  [4,5,6]]
B = torch.arange(1., 13.).reshape(3, 4)

show('A', A, values=True)
show('B', B, values=True)

m, k = A.shape          # 2, 3
k2, n = B.shape         # 3, 4
print(f'\nm={m}  k={k}   k={k2}  n={n}   -> inner dims {k} and {k2} match: {k == k2}')

A                        shape=(2, 3)  dtype=torch.float32  device=cpu
tensor([[1., 2., 3.],
        [4., 5., 6.]])
B                        shape=(3, 4)  dtype=torch.float32  device=cpu
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])

m=2  k=3   k=3  n=4   -> inner dims 3 and 3 match: True


In [3]:
# Way 1: PyTorch
C_torch = A @ B

# Way 2: one dot product per output cell, written out longhand
C_loop = torch.zeros(m, n)
for i in range(m):              # every row of A
    for j in range(n):          # every column of B
        row_i = A[i, :]         # shape (3,)  <- 1-D
        col_j = B[:, j]         # shape (3,)  <- 1-D
        C_loop[i, j] = torch.dot(row_i, col_j)   # the 00.3 operation

show('C_torch', C_torch, values=True)
show('C_loop ', C_loop,  values=True)
print('\nidentical:', torch.allclose(C_torch, C_loop))
print('number of dot products performed:', m * n)

C_torch                  shape=(2, 4)  dtype=torch.float32  device=cpu
tensor([[ 38.,  44.,  50.,  56.],
        [ 83.,  98., 113., 128.]])
C_loop                   shape=(2, 4)  dtype=torch.float32  device=cpu
tensor([[ 38.,  44.,  50.,  56.],
        [ 83.,  98., 113., 128.]])

identical: True
number of dot products performed: 8


### Reading the output
The two agree exactly. `(2, 3) @ (3, 4) -> (2, 4)`, which is **2 × 4 = 8 cells**, so
**8 dot products**, each over vectors of length 3.

Check one by hand: `C[0, 0] = row 0 of A · column 0 of B = (1×1) + (2×5) + (3×9) = 38`.

```
(2, 3) @ (3, 4)  ->  (2, 4)
 ↑   └────┬────┘      ↑
 │   inner dims       │
 │   match, then      │
 │   VANISH           │
 └──── outer dims survive ────┘
```

The `3` is the *length of the vectors being dotted*. Once you sum them it is gone.

## Experiment 2 — the cost

Each output cell costs `k` multiplications and `k − 1` additions, so `2k − 1` operations.
There are `m · n` cells.

In [4]:
exact   = m * n * (2 * k - 1)
rounded = 2 * m * k * n

print(f'cells            : m*n         = {m}*{n} = {m*n}')
print(f'ops per cell     : 2k-1        = 2*{k}-1 = {2*k-1}   ({k} mults + {k-1} adds)')
print(f'exact total      : m*n*(2k-1)  = {exact}')
print(f'rounded (quoted) : 2*m*k*n     = {rounded}')

cells            : m*n         = 2*4 = 8
ops per cell     : 2k-1        = 2*3-1 = 5   (3 mults + 2 adds)
exact total      : m*n*(2k-1)  = 40
rounded (quoted) : 2*m*k*n     = 48


### Reading the output
`40` exact versus `48` rounded. For `k = 3` the gap is visible; at `k = 4096` the
`−1` is noise, which is why everyone quotes `2mkn`. Knowing *why* it is `2` and not `1`
(a multiply **and** an add) is the interview detail.

## Experiment 3 — when it is illegal, and what the error says

`A` is `(2, 3)` and `B` is `(3, 4)`. Flip the order: `B @ A` is `(3, 4) @ (2, 3)`.
Inner dims are `4` and `2`. They do not match, so there is nothing to dot.

**Matmul is not commutative** — `A @ B` and `B @ A` are different operations, and often
one of them does not even exist.

In [5]:
try:
    B @ A
except RuntimeError as e:
    print('B @ A ->', type(e).__name__)
    print('   ', str(e).split('\n')[0])

# and the case from the lesson: (1,3) @ (1,2)
try:
    torch.ones(1, 3) @ torch.ones(1, 2)
except RuntimeError as e:
    print('\n(1,3) @ (1,2) ->', str(e).split('\n')[0])

# fix the inner dim and it works
print('\n(1,3) @ (3,2) ->', (torch.ones(1, 3) @ torch.ones(3, 2)).shape)

B @ A -> RuntimeError
    mat1 and mat2 shapes cannot be multiplied (3x4 and 2x3)

(1,3) @ (1,2) -> mat1 and mat2 shapes cannot be multiplied (1x3 and 1x2)

(1,3) @ (3,2) -> torch.Size([1, 2])


### Reading the output
PyTorch prints **both shapes** in the error, so you can see immediately which pair failed.
Learn to read that line — it is the most common error you will hit all year.

## Experiment 4 — 1-D is neither a row nor a column

On paper, people draw a vector as a row or a column and treat them as the same thing.
**PyTorch does not.** Count the numbers in the shape and you know what you have.

In [6]:
v = torch.tensor([3., 4.])

print('1-D   :', v, ' shape', tuple(v.shape), ' ndim', v.ndim)
print('        -> ONE number in the shape. A flat list. No rows, no columns.\n')

row = v.unsqueeze(0)    # add an axis at the FRONT -> (1, 2)
col = v.unsqueeze(1)    # add an axis at the BACK  -> (2, 1)

print('row   :', tuple(row.shape), 'ndim', row.ndim)
print(row)
print('\ncolumn:', tuple(col.shape), 'ndim', col.ndim)
print(col)
print('\nThe extra bracket IS the extra dimension.')

1-D   : tensor([3., 4.])  shape (2,)  ndim 1
        -> ONE number in the shape. A flat list. No rows, no columns.

row   : (1, 2) ndim 2
tensor([[3., 4.]])

column: (2, 1) ndim 2
tensor([[3.],
        [4.]])

The extra bracket IS the extra dimension.


### Reading the output

| What you see | numbers in shape | `ndim` | what it is |
| --- | ---: | ---: | --- |
| `torch.Size([2])` | 1 | 1 | flat list — neither row nor column |
| `torch.Size([1, 2])` | 2 | 2 | **row** vector |
| `torch.Size([2, 1])` | 2 | 2 | **column** vector |

`torch.dot` accepts **1-D only**. The moment you are 2-D you use `@` and the inner/outer
rule applies. Different tools for different `ndim`.

## Experiment 5 — leading dims are batches (a free for-loop)

`(2, 4, 8) @ (8, 16)`. Matmul only ever looks at the **last two** dimensions.
Anything in front is a batch: *do this same operation once per item*.

The leading dim is not ignored — it is the **count of independent problems**.

In [7]:
Ab = torch.randn(2, 4, 8)     # 2 separate (4,8) matrices
Wb = torch.randn(8, 16)       # ONE (8,16), shared by both

shape_table({'Ab': Ab, 'Wb': Wb, 'Ab @ Wb': Ab @ Wb})
print()

# the same thing written as an explicit loop
out = torch.empty(2, 4, 16)
for i in range(2):
    out[i] = Ab[i] @ Wb
    print(f'  loop {i}: Ab[{i}] {tuple(Ab[i].shape)} @ Wb {tuple(Wb.shape)} -> {tuple(out[i].shape)}')

print('\nloop == batched:', torch.allclose(out, Ab @ Wb))

tensor  | shape
--------+------
Ab      | (2, 4, 8)
Wb      | (8, 16)
Ab @ Wb | (2, 4, 16)

  loop 0: Ab[0] (4, 8) @ Wb (8, 16) -> (4, 16)
  loop 1: Ab[1] (4, 8) @ Wb (8, 16) -> (4, 16)

loop == batched: True


### Reading the output
`(2, 4, 8) @ (8, 16) -> (2, 4, 16)`. The `8`s matched and vanished; `4` and `16` survived;
the leading `2` rode along untouched.

**Why batch rather than loop?** From 00.2 you already measured it: handing a job to the GPU
has overhead, and for a small matmul that overhead dominates. Two separate calls pay it
twice and leave most of the GPU idle. One batched call pays it once and fills the hardware.
Same maths, very different cost — the JDBC batch-insert argument.

## Experiment 6 — a batch is the SAME weights on DIFFERENT data

The common misreading is "it multiplies the same matrix twice, why bother?".
It does not. The data differs per batch item; only the weight is shared.

Think: two user requests arriving at one model.

In [8]:
Ad = torch.tensor([
    [[1., 0., 0.],      # request 0
     [0., 1., 0.]],

    [[5., 5., 5.],      # request 1  <- different numbers
     [9., 9., 9.]],
])

Wd = torch.tensor([     # ONE weight. This is the model.
    [1., 2.],
    [3., 4.],
    [5., 6.],
])

outd = Ad @ Wd
print('Ad', tuple(Ad.shape), ' Wd', tuple(Wd.shape), ' -> out', tuple(outd.shape))
print('\nout[0] =\n', outd[0])
print('\nout[1] =\n', outd[1])
print('\nidentical results?', torch.equal(outd[0], outd[1]))

Ad (2, 2, 3)  Wd (3, 2)  -> out (2, 2, 2)

out[0] =
 tensor([[1., 2.],
        [3., 4.]])

out[1] =
 tensor([[ 45.,  60.],
        [ 81., 108.]])

identical results? False


### Reading the output
`False` — different inputs give different outputs. Nobody multiplies the same matrix twice;
they push **different data** through the **same weights**, together, to keep an expensive GPU fed.

In a real model `A` is `(B, T, d)`: `B` sequences, `T` tokens each, `d` features per token.
Raising `B` raises throughput — and latency, and memory, because every in-flight sequence
holds its own KV cache. That trade-off is Phase 5.

## Experiment 7 — the output's last dim comes from the WEIGHT

The most common misconception: that the last dimension of the output comes from your data.
It comes from `W`. Read `W` of shape `(d_in, d_out)` as a converter:
**takes `d_in` features in, produces `d_out` features out**, leaving every leading dim alone.

In [9]:
x = torch.randn(2, 4, 8)          # B=2 sequences, T=4 tokens, d_in=8 features

for d_out in (16, 32, 3):
    W = torch.randn(8, d_out)     # only d_out changes
    print(f'x {tuple(x.shape)} @ W {tuple(W.shape)} -> {tuple((x @ W).shape)}'
          f'    (B and T untouched, features 8 -> {d_out})')

x (2, 4, 8) @ W (8, 16) -> (2, 4, 16)    (B and T untouched, features 8 -> 16)
x (2, 4, 8) @ W (8, 32) -> (2, 4, 32)    (B and T untouched, features 8 -> 32)
x (2, 4, 8) @ W (8, 3) -> (2, 4, 3)    (B and T untouched, features 8 -> 3)


### Reading the output
`B=2` and `T=4` never move. Only the feature count changes, and it changes to whatever
`W`'s second dimension says. That single reading explains nearly every matmul in a
Transformer — including `d_model -> 4·d_model` in a feed-forward block.

## Challenge — predict first, then run

Write your answers **before** running the cell.

| # | Expression | Legal? | Output shape | Which dim vanished? |
| --- | --- | --- | --- | --- |
| 1 | `(5, 7) @ (7, 2)` | _(write)_ | _(write)_ | _(write)_ |
| 2 | `(5, 7) @ (2, 7)` | _(write)_ | _(write)_ | _(write)_ |
| 3 | `(8, 3, 4) @ (4, 10)` | _(write)_ | _(write)_ | _(write)_ |
| 4 | `(3,) @ (3,)` | _(write)_ | _(write)_ | _(write)_ |

And: how many operations does #1 cost, exactly, using `m · n · (2k − 1)`?

In [ ]:
# run AFTER writing your predictions above
cases = [
    ((5, 7), (7, 2)),
    ((5, 7), (2, 7)),
    ((8, 3, 4), (4, 10)),
    ((3,), (3,)),
]

for sa, sb in cases:
    try:
        result = torch.ones(*sa) @ torch.ones(*sb)
        print(f'{str(sa):12s} @ {str(sb):10s} -> {tuple(result.shape)}'
              f'{"   (scalar)" if result.ndim == 0 else ""}')
    except RuntimeError as e:
        print(f'{str(sa):12s} @ {str(sb):10s} -> FAILED: {str(e).split(chr(10))[0]}')